In [0]:
CREATE OR REPLACE VIEW airline_catalog.semantic.vw_airport_origin_performance AS -- Crear o reemplazar vista de desempeño aeroportuario

WITH airport_metrics AS -- CTE para métricas de aeropuerto
(
    SELECT

        da.airport_id, -- ID del aeropuerto
        da.airport_code, -- Código del aeropuerto
        da.airport_name, -- Nombre del aeropuerto

        CONCAT_WS(', ', da.city, da.state) AS location, -- Ubicación (Ciudad, Estado)

        COUNT(*) AS total_departures, -- Total de salidas

        ROUND(AVG(ff.departure_delay),2) AS avg_departure_delay, -- Promedio de retraso en salida

        SUM(CASE
                WHEN ff.cancelled THEN 1 -- Si el vuelo fue cancelado
                ELSE 0 -- Si no fue cancelado
            END) AS cancelled_departures, -- Total de salidas canceladas

        ROUND(
            100.0 *
            SUM(CASE WHEN ff.cancelled THEN 1 ELSE 0 END) -- Total de cancelaciones
            / COUNT(*), -- Total de salidas
            2
        ) AS pct_cancelled, -- Porcentaje de salidas canceladas
        ROUND(AVG(ff.distance),2) AS avg_distance, -- Promedio de distancia recorrida
        SUM(ff.distance) AS total_distance -- Total de distancia recorrida

    FROM airline_catalog.gold.fact_flights ff -- Tabla de hechos de vuelos

    INNER JOIN airline_catalog.gold.dim_airport da -- Unión con dimensión aeropuerto
        ON ff.origin_airport_sk = da.airport_id -- Condición de unión por aeropuerto de origen

    GROUP BY

        da.airport_id, -- Agrupar por ID de aeropuerto
        da.airport_code, -- Agrupar por código de aeropuerto
        da.airport_name, -- Agrupar por nombre de aeropuerto
        da.city, -- Agrupar por ciudad
        da.state -- Agrupar por estado
)

SELECT

    *, -- Seleccionar todas las columnas

    RANK() OVER (
        ORDER BY total_departures DESC -- Ranking por tráfico (salidas)
    ) AS traffic_rank, -- Ranking de tráfico

    RANK() OVER (
        ORDER BY avg_departure_delay ASC -- Ranking por puntualidad (menor retraso)
    ) AS punctuality_rank -- Ranking de puntualidad

FROM airport_metrics; -- Fuente: CTE de métricas de aeropuerto